# BDS Việt Nam — Phân tích Chuyên sâu Thị trường Bất động sản

> **Generated by AI Lab DataHub** · `bds-phan-tich-chuyen-sau` v1.0.0 · 2026-06-12T09:44:26.105828+00:00

Phân tích chuyên sâu thị trường BĐS thuần (filter-first, 12/24 báo cáo BĐS): cung-cầu-giá theo quý 2024Q1–2026Q1, 6 figures, kèm star schema 3 fact (supply/demand/price) + 4 dim (date/region/segment/report) tải về được. Notebook self-contained — chạy ngay trên Colab không cần API key.

**License**: CC-BY-4.0  
**Citation**: UEL FinTech Lab (2026). BDS Việt Nam — Phân tích Chuyên sâu Thị trường Bất động sản. DataHub n2nai.  
**Tags**: bat-dong-san, real-estate, vietnam, star-schema, fact-dim  
**Runtime**: 🟢 Colab Free tier OK


## Quick start

1. **Runtime → Run all** — notebook đã có sẵn API Key (auto-injected khi publish).
2. Nếu cần rotate key: vào [DataHub Workspace](https://datahub.n2nai.io/workspace) → Project `uel-fintech` → tab **API Gateway**.

_Không cần tạo Colab Secret — key đã embed trong Cell 1._


## Cell 1 — Setup + auth check


In [ ]:
!pip install -q requests pandas pyarrow
import os, io, requests, pandas as pd

BASE_URL = "https://datahub.n2nai.io"
PROJECT_ID = 29
PROJECT_NAME = "uel-fintech"
PRODUCT_SLUG = "bds-phan-tich-chuyen-sau"

# ── DataHub API Key (auto-injected by publish — rotate qua UI Workspace → API Keys) ──
DATAHUB_API_KEY = "prj_uel-fintech_gvWxrnTaFpfyhnCUzt2qLD-6mNqq8822jLt8s7yKK1Q"
HEADERS = {'X-API-Key': DATAHUB_API_KEY}

# Verify key OK + show scope
r = requests.get(f'{BASE_URL}/api/v1/projects/{PROJECT_NAME}/api-keys/whoami', headers=HEADERS)
print('Auth status:', r.status_code)
print(r.json() if r.ok else r.text)

## Source notebook content

Inlined from `BDS/01_phan_tich_chuyen_sau_BDS.ipynb` — version `1.0.0`.

Bao gồm 17 cell gốc từ notebook owner đã publish.


# 🏢 Phân tích chuyên sâu Thị trường Bất động sản Việt Nam
### Từ kho 24 báo cáo REFI/AFA — phương pháp **lọc trước, phân tích sau** (filter-first)

> **Slide note — Phương pháp luận (1 slide):**
> Thay vì quét toàn bộ 24 báo cáo lấy hết mọi data (vĩ mô + BĐS), pipeline bước ⑧:
> 1. **Lọc báo cáo**: phân loại 24 báo cáo → giữ **13 báo cáo liên quan BĐS** (12 báo cáo *"Thị trường Bất động sản"* 2025-05→2026-05 + 1 báo cáo *Chiến lược 02/2026* chứa chương BĐS nguồn Bộ Xây dựng), **loại 11 báo cáo vĩ mô tháng**.
> 2. **Lọc data**: từ master dataset chỉ giữ `domain = real_estate` (284/522 quan sát) → `fact_real_estate.csv` thuần BĐS, bỏ 9 chỉ số vĩ mô.
> 3. **Phân tích chuyên sâu** trên tập BĐS: cung — cầu — tồn kho — giá — chỉ số phái sinh (hấp thụ, QoQ).
>
> Lợi ích: tập trung đúng domain, không nhiễu vĩ mô, chi phí khai thác thấp, lineage rõ (mỗi số đều trace về báo cáo + trang PDF).

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 110, 'axes.unicode_minus': False})
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
DS = 'datasets'

# Catalog đã phân loại — minh chứng bước LỌC BÁO CÁO
cat = pd.read_csv(f'{DS}/reports_catalog_classified.csv')
print('Phân loại 24 báo cáo:')
print(cat['category'].value_counts().to_string())
re_cat = pd.read_csv(f'{DS}/re_reports_catalog.csv')
print(f"\n→ {len(re_cat)} báo cáo BĐS được giữ lại cho phân tích:")
for _, r in re_cat.iterrows():
    print(f"  • {r['report_month']}  {r['file'][:60]}")

## 1. Inventory dữ liệu BĐS thuần (sau lọc)
**Slide note:** 284 quan sát BĐS / 522 tổng — chia 12 nhóm chỉ số; fact table BĐS `fact_real_estate.csv` chỉ giữ measures BĐS, bỏ hẳn 9 cột vĩ mô.

In [ ]:
re_long = pd.read_csv(f'{DS}/re_master_long.csv')
fact = pd.read_csv(f'{DS}/fact_real_estate.csv')
print(f"re_master_long.csv : {re_long.shape[0]} quan sát BĐS (100% domain=real_estate)")
print(f"fact_real_estate   : {fact.shape[0]} dòng × {fact.shape[1]} cột")
summary = re_long.groupby('metric').agg(n=('value','count'),
    confidence_high=('confidence', lambda s: (s=='high').mean()*100)).round(0)
summary.sort_values('n', ascending=False)

## 2. Nguồn cung nhà ở toàn quốc (theo quý — Bộ Xây dựng)
**Slide note:** Sản phẩm đang triển khai ~400 nghìn; nguồn "đủ điều kiện mở bán" và "hoàn thành" phản ánh cung thực tế ra thị trường từng quý.

In [ ]:
sup = pd.read_csv(f'{DS}/re_supply_quarterly.csv')
piv = sup.pivot_table(index='quarter', columns='metric', values='value', aggfunc='sum')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
small = [c for c in ['du_dieu_kien_mo_ban','da_hoan_thanh','new_supply'] if c in piv.columns]
piv[small].plot(ax=axes[0], marker='o')
axes[0].set_title('Cung thực tế ra thị trường theo quý (căn)')
axes[0].set_ylabel('căn'); axes[0].grid(alpha=.3); axes[0].legend(loc='best', fontsize=9)
if 'products_under_development' in piv.columns:
    piv['products_under_development'].dropna().plot(ax=axes[1], marker='s', color='tab:red')
    axes[1].set_title('Sản phẩm đang triển khai (tồn pipeline)')
    axes[1].set_ylabel('sản phẩm'); axes[1].grid(alpha=.3)
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print(piv.round(0).to_string())

## 3. Nguồn cung mới theo thành phố (HN / ĐN / TP.HCM, 2022Q1→2025Q3)
**Slide note:** Chuỗi 15 quý cross-validated 100% giữa 2 báo cáo quý (08/2025 & 11/2025). Chung cư HN dẫn dắt chu kỳ phục hồi 2024-2025; TP.HCM phục hồi chậm hơn.

In [ ]:
city = pd.read_csv(f'{DS}/re_supply_city_quarterly.csv')
cq = city[city['granularity']=='quarter'].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
for cname, g in cq.groupby('city'):
    g = g.sort_values('period')
    axes[0].plot(g['period'], g['chung_cu'], marker='o', label=cname)
    axes[1].plot(g['period'], g['bds_gan_lien_dat'], marker='s', label=cname)
axes[0].set_title('Nguồn cung mới — Chung cư (căn/quý)')
axes[1].set_title('Nguồn cung mới — BĐS gắn liền đất (căn/quý)')
for ax in axes:
    ax.legend(fontsize=9); ax.grid(alpha=.3); ax.tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()
# QoQ growth chuyên sâu
hn = cq[cq.city=='Hà Nội'].sort_values('period')
print('QoQ tăng trưởng cung chung cư Hà Nội (%):')
print((hn.set_index('period')['chung_cu'].pct_change()*100).round(1).dropna().to_string())

## 4. Giao dịch — quý & năm
**Slide note:** Đất nền chiếm ~3/4 lượng giao dịch; tổng giao dịch 2022 (785K) sụt mạnh 2023 (434K) rồi phục hồi. Quý 2025: thanh khoản đất nền vượt trội chung cư + nhà riêng lẻ.

In [ ]:
tq = pd.read_csv(f'{DS}/re_transactions_quarterly.csv')
ta = pd.read_csv(f'{DS}/re_transactions_annual.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pt = tq.pivot_table(index='quarter', columns='segment', values='transactions', aggfunc='sum')
pt.plot(kind='bar', ax=axes[0], width=.8)
axes[0].set_title('Giao dịch theo quý & phân khúc'); axes[0].set_ylabel('giao dịch')
axes[0].legend(fontsize=8); axes[0].grid(alpha=.3, axis='y'); axes[0].tick_params(axis='x', rotation=45)
ta_p = ta.set_index('year')[['chungcu_nha_rieng_le','dat_nen']]
ta_p.plot(kind='bar', stacked=True, ax=axes[1], color=['tab:blue','tab:orange'])
axes[1].set_title('Giao dịch theo năm (stacked)'); axes[1].set_ylabel('giao dịch')
axes[1].legend(['Chung cư + nhà riêng lẻ','Đất nền'], fontsize=9); axes[1].grid(alpha=.3, axis='y')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()
share = (ta['dat_nen']/ta['total']*100).round(1)
print('Tỷ trọng đất nền trong tổng giao dịch (%):', dict(zip(ta['year'], share)))

## 5. Tồn kho BĐS theo quý & phân khúc
**Slide note:** Tồn kho (số liệu Bộ Xây dựng) — theo dõi chu kỳ hấp thụ của từng phân khúc.

In [ ]:
inv = pd.read_csv(f'{DS}/re_inventory_quarterly.csv')
pi = inv.pivot_table(index='quarter', columns='segment', values='inventory_units', aggfunc='sum')
ax = pi.plot(marker='o', figsize=(11, 4.5))
ax.set_title('Tồn kho BĐS theo quý & phân khúc (căn/lô/nền)')
ax.set_ylabel('căn/lô/nền'); ax.grid(alpha=.3); ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print(pi.round(0).tail(6).to_string())

## 6. Mặt bằng giá theo vùng & phân khúc
**Slide note:** Giá sơ cấp 2025 tăng mạnh nhất ở chung cư miền Bắc (+20-30% YoY); Q1/2026 đa số vùng đi ngang hoặc tăng nhẹ 1-2%.

In [ ]:
p25 = pd.read_csv(f'{DS}/re_price_range_2025.csv')
p26 = pd.read_csv(f'{DS}/re_price_q1_2026.csv')
fig, ax = plt.subplots(figsize=(12, 5.5))
lab = p25['region'] + ' · ' + p25['segment']
y = np.arange(len(p25))
ax.hlines(y, p25['price_min_tr_m2'], p25['price_max_tr_m2'], color='tab:blue', lw=5, alpha=.7)
ax.plot(p25['price_min_tr_m2'], y, '|', color='navy', ms=10)
ax.plot(p25['price_max_tr_m2'], y, '|', color='navy', ms=10)
ax.set_yticks(y); ax.set_yticklabels(lab, fontsize=8)
ax.set_xlabel('triệu đồng/m²'); ax.set_title('Dải giá sơ cấp 2025 theo vùng × phân khúc (min–max)')
ax.grid(alpha=.3, axis='x')
plt.tight_layout(); plt.show()
print('Biến động QoQ Q1/2026 theo vùng × phân khúc:')
print(p26.groupby(['region','segment'])['trend_qoq'].first().to_string())

## 7. Phân tích chuyên sâu — chỉ số phái sinh
**Slide note (2 slide):**
- **Sức hấp thụ**: so giao dịch chung cư + nhà riêng lẻ với nguồn "đủ điều kiện mở bán" cùng quý — đo cầu thực so cung mới.
- **Cơ cấu TP.HCM mới sau sáp nhập** (Q3/2025): Bình Dương cũ đóng góp ~75% nguồn cung căn hộ vùng TP.HCM mở rộng.

In [ ]:
# 7a. Sức hấp thụ: transactions (chung cư + nhà riêng lẻ) / du_dieu_kien_mo_ban
mo_ban = sup[sup.metric=='du_dieu_kien_mo_ban'].set_index('quarter')['value']
gd_cc = tq[tq.segment.str.contains('chung_cu', na=False)].groupby('quarter')['transactions'].sum()
common = mo_ban.index.intersection(gd_cc.index)
absorb = pd.DataFrame({'cung_mo_ban': mo_ban[common], 'giao_dich_cc_nha': gd_cc[common]})
absorb['ty_le_hap_thu'] = (absorb['giao_dich_cc_nha']/absorb['cung_mo_ban']).round(2)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
absorb[['cung_mo_ban','giao_dich_cc_nha']].plot(kind='bar', ax=axes[0], width=.75)
axes[0].set_title('Cung đủ ĐK mở bán vs Giao dịch CC+nhà riêng lẻ')
axes[0].legend(['Cung mở bán','Giao dịch'], fontsize=9); axes[0].grid(alpha=.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)
# 7b. Cơ cấu TP.HCM mới
hcm = pd.read_csv(f'{DS}/re_supply_hcm_region_2025q3.csv')
vals = hcm.set_index('region')['can_ho_chung_cu']
vals = vals[vals > 0]
axes[1].pie(vals, labels=vals.index, autopct='%1.0f%%', startangle=90,
            colors=['tab:blue','tab:orange','tab:green'][:len(vals)])
axes[1].set_title('Cơ cấu cung căn hộ TP.HCM mới — Q3/2025 (sau sáp nhập)')
plt.tight_layout(); plt.show()
print('Tỷ lệ hấp thụ (giao dịch / cung mở bán) theo quý:')
print(absorb['ty_le_hap_thu'].to_string())

## 8. Kết luận & insight chính
**Slide note (slide chốt):**
1. **Filter-first hiệu quả**: 13/24 báo cáo + 284/522 quan sát đủ cho toàn bộ phân tích BĐS — không cần quét 11 báo cáo vĩ mô.
2. **Cung**: pipeline ~400K sản phẩm đang triển khai; cung thực tế mở bán theo quý phục hồi rõ từ 2024; Hà Nội dẫn dắt chu kỳ chung cư.
3. **Cầu**: đất nền chiếm ~70-80% lượng giao dịch; thanh khoản phục hồi từ đáy 2023.
4. **Giá**: chung cư miền Bắc tăng nóng nhất 2025 (+20-30% YoY), Q1/2026 hạ nhiệt còn +1-2% QoQ.
5. **Cơ cấu mới**: TP.HCM mở rộng — Bình Dương cũ thành động lực cung chính (~75% căn hộ).
6. **Lineage**: mọi số liệu trace về báo cáo + trang PDF (cột `source_report`/`source_page`).

## Reproducibility lock

Notebook này pin tới Data Product version `1.0.0` — kết quả reproducible bất kể source data thay đổi sau này.


In [ ]:
print('DataHub Product: bds-phan-tich-chuyen-sau v1.0.0')
print('Generated: 2026-06-12T09:44:26.105828+00:00')
print(f'Pandas: {pd.__version__}')